# Embedding Analysis — VQ Regularization Evidence

Empirically supports the claim that VQ acts as a representation regularizer.

**What this produces:**
1. **Silhouette scores** per method — higher = tighter within-class, better-separated between-class
2. **t-SNE plots** — A_plain_vfl vs H_vq_K64 side by side, colored by class
3. **Codebook utilization** — fraction of K entries used per VQ method (sanity check for codebook collapse)

**Attach datasets:** same as main notebook (results_final + isic-2019-256-cache)

**Download:** `/kaggle/working/embedding_analysis/`

In [ ]:
import subprocess, sys
for pkg in ['scikit-learn', 'timm', 'umap-learn']:
    mod = {'scikit-learn': 'sklearn', 'umap-learn': 'umap'}.get(pkg, pkg)
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
print('Environment ready.')

In [ ]:
import os, json, shutil
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, List, Tuple, Any
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import timm

print(f'torch {torch.__version__}  cuda={torch.cuda.is_available()}')

In [ ]:
# ── FILL IN the dataset path below (check your Kaggle Data sidebar) ──
RESULTS_DATASET = "/kaggle/input/YOUR-RESULTS-DATASET-SLUG/results_final"

WORKING  = "/kaggle/working"
CKPT_DIR = f"{WORKING}/checkpoints"
OUT_DIR  = f"{WORKING}/embedding_analysis"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(OUT_DIR,  exist_ok=True)

if not os.path.exists(RESULTS_DATASET):
    raise FileNotFoundError(f"Dataset not found: {RESULTS_DATASET}")

ckpt_src = os.path.join(RESULTS_DATASET, 'checkpoints')
copied = 0
for fn in os.listdir(ckpt_src):
    if fn.endswith('.pt'):
        shutil.copy(os.path.join(ckpt_src, fn), CKPT_DIR)
        copied += 1
print(f'Copied {copied} checkpoints.')

In [ ]:
CACHE_PATH = "/kaggle/input/datasets/taimurjahanzeb/isic-2019-256-cache/images_256.npy"
FOLD_PATH  = "/kaggle/input/datasets/taimurjahanzeb/isic-2019-256-cache/fold_indices.npz"
GT_CSV     = "/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_GroundTruth.csv"
META_CSV   = "/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Metadata.csv"

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

@dataclass
class V9Config:
    input_size: int             = 224
    store_size: int             = 256
    batch_size: int             = 64
    fold: int                   = 0
    vq_proj_dim: int            = 128
    vq_num_subspaces: int       = 8
    vq_codebook_size: int       = 256
    vq_commitment_weight: float = 0.25
    vq_codebook_weight: float   = 1.0
    vq_warmup_epochs: int       = 3
    vq_use_kmeans_init: bool    = True
    kmeans_samples: int         = 4096
    rm_sigma_fwd: float         = 0.01
    rm_sigma_bwd: float         = 0.01
    usage_entropy_weight: float = 0.01
    seeds: Tuple[int, ...]      = (42, 43)
    num_workers: int            = 2

CFG = V9Config()
print(f'Device: {DEVICE}')

In [ ]:
gt = pd.read_csv(GT_CSV).drop(columns=['UNK'], errors='ignore')
CLASS_NAMES = gt.columns[1:].tolist()
NUM_CLASSES = len(CLASS_NAMES)

meta_df = pd.read_csv(META_CSV)
merged  = gt.merge(meta_df, on='image', how='inner')

bins       = list(range(0, 90, 5))
age_labels = [f'age_{b}_{b+4}' for b in bins[:-1]] + ['age_85plus']
merged['age_bin'] = pd.cut(merged['age_approx'], bins=bins+[np.inf],
                           labels=age_labels, right=False).astype(str)
merged.loc[merged['age_approx'].isna(), 'age_bin'] = 'age_unknown'
age_oh = pd.get_dummies(merged['age_bin']).astype(np.float32).values

site_col = ('anatom_site_general_challenge'
            if 'anatom_site_general_challenge' in merged.columns
            else 'anatom_site_general')
merged[site_col] = merged[site_col].fillna('unknown')
loc_oh = pd.get_dummies(merged[site_col]).astype(np.float32).values
merged['sex'] = merged['sex'].fillna('unknown')
sex_oh = pd.get_dummies(merged['sex']).astype(np.float32).values

labels_array = merged[CLASS_NAMES].values.astype(np.float32)
label_ints   = np.argmax(labels_array, axis=1)
class_counts = np.bincount(label_ints, minlength=NUM_CLASSES).astype(np.float32)
tail_classes = sorted(np.argsort(class_counts)[:4].tolist())

_fd        = np.load(FOLD_PATH)
train_ind  = _fd[f'train_{CFG.fold}']
val_ind    = _fd[f'val_{CFG.fold}']
all_images = np.load(CACHE_PATH)

SET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
SET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
val_transforms = transforms.Compose([
    transforms.CenterCrop(CFG.input_size),
    transforms.ToTensor(),
    transforms.Normalize(SET_MEAN, SET_STD),
])

print(f'Val samples: {len(val_ind)}  Classes: {CLASS_NAMES}')

In [ ]:
class V8Backbone(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.net = timm.create_model('efficientnet_b0', pretrained=pretrained, num_classes=0)
        self.out_dim = 1280
    def forward(self, x): return self.net(x)

class V8Projection(nn.Module):
    def __init__(self, in_dim=1280, out_dim=128):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.bn     = nn.BatchNorm1d(out_dim)
    def forward(self, h): return self.bn(self.linear(h))

class V8ProductQuantizer(nn.Module):
    def __init__(self, proj_dim, num_subspaces, codebook_size, commitment_weight=0.25):
        super().__init__()
        assert proj_dim % num_subspaces == 0
        self.proj_dim = proj_dim; self.M = num_subspaces
        self.K = codebook_size; self.subdim = proj_dim // num_subspaces
        self.commitment_weight = commitment_weight
        self.codebooks = nn.Parameter(torch.randn(num_subspaces, codebook_size, self.subdim)*0.02)
        self.register_buffer('usage_counts', torch.zeros(num_subspaces, codebook_size, dtype=torch.long))
    def forward(self, z):
        B, D = z.shape
        z_sub = z.view(B, self.M, self.subdim)
        distances = (z_sub.unsqueeze(2).pow(2).sum(-1)
                     + self.codebooks.pow(2).sum(-1).unsqueeze(0)
                     - 2.0 * torch.einsum('bmd,mkd->bmk', z_sub, self.codebooks))
        indices = distances.argmin(dim=-1)
        q_sub = torch.gather(self.codebooks.unsqueeze(0).expand(B,-1,-1,-1), 2,
                             indices.unsqueeze(-1).unsqueeze(-1).expand(-1,-1,1,self.subdim)).squeeze(2)
        q = (z_sub + (q_sub - z_sub).detach()).reshape(B, D)
        with torch.no_grad():
            for m in range(self.M):
                self.usage_counts[m] += torch.bincount(indices[:, m], minlength=self.K)
        return q, indices, torch.zeros((), device=z.device)

class PlainPassiveClient(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = V8Backbone(pretrained=True); self.passive_emb_dim = 1280
    def forward(self, x, use_quantizer=True):
        h = self.backbone(x); return h, None, torch.zeros((), device=h.device), h

class ProjPassiveClient(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = V8Backbone(pretrained=True)
        self.projection = V8Projection(1280, cfg.vq_proj_dim); self.passive_emb_dim = cfg.vq_proj_dim
    def forward(self, x, use_quantizer=True):
        z = self.projection(self.backbone(x)); return z, None, torch.zeros((), device=z.device), z

class SignPassiveClient(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = V8Backbone(pretrained=True)
        self.projection = V8Projection(1280, cfg.vq_proj_dim); self.passive_emb_dim = cfg.vq_proj_dim
    def forward(self, x, use_quantizer=True):
        z = self.projection(self.backbone(x))
        s = z + (z.sign() - z).detach(); return s, None, torch.zeros((), device=z.device), z

class RandSignPassiveClient(nn.Module):
    def __init__(self, cfg, init_seed=0):
        super().__init__()
        self.backbone = V8Backbone(pretrained=True)
        W = torch.randn(1280, cfg.vq_proj_dim, generator=torch.Generator().manual_seed(int(init_seed))) / (1280**0.5)
        self.register_buffer('proj_W', W); self.passive_emb_dim = cfg.vq_proj_dim
    def forward(self, x, use_quantizer=True):
        z = self.backbone(x) @ self.proj_W
        s = z + (z.sign() - z).detach(); return s, None, torch.zeros((), device=z.device), z

class VQPassiveClient(nn.Module):
    def __init__(self, cfg, codebook_size=None, num_subspaces=None, commitment_weight=None):
        super().__init__()
        K = codebook_size or cfg.vq_codebook_size
        M = num_subspaces or cfg.vq_num_subspaces
        beta = commitment_weight if commitment_weight is not None else cfg.vq_commitment_weight
        self.backbone   = V8Backbone(pretrained=True)
        self.projection = V8Projection(1280, cfg.vq_proj_dim)
        self.quantizer  = V8ProductQuantizer(cfg.vq_proj_dim, M, K, beta)
        self.passive_emb_dim = cfg.vq_proj_dim
    def forward(self, x, use_quantizer=True):
        z = self.projection(self.backbone(x))
        if not use_quantizer: return z, None, torch.zeros((), device=z.device), z
        q, idx, vq_loss = self.quantizer(z); return q, idx, vq_loss, z

@dataclass
class MethodSpec:
    name: str
    passive_type: str
    comm_bits: int
    vq_codebook_size: int = 256; vq_num_subspaces: int = 8
    vq_commitment_weight: float = 0.25; vq_use_kmeans_init: bool = True
    invernet_epochs: int = 50

def build_passive(spec, seed):
    cfg = CFG
    if spec.passive_type == 'plain':     return PlainPassiveClient(cfg)
    if spec.passive_type == 'proj':      return ProjPassiveClient(cfg)
    if spec.passive_type == 'sign':      return SignPassiveClient(cfg)
    if spec.passive_type == 'rand_sign': return RandSignPassiveClient(cfg, init_seed=seed)
    if spec.passive_type == 'vq':
        return VQPassiveClient(cfg, codebook_size=spec.vq_codebook_size,
                               num_subspaces=spec.vq_num_subspaces,
                               commitment_weight=spec.vq_commitment_weight)
    raise ValueError(spec.passive_type)

def _torch_load(path):
    try: return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError: return torch.load(path, map_location=DEVICE)

def _build_passive_for_load(spec_dict, seed):
    s = MethodSpec(name=spec_dict['name'], passive_type=spec_dict['passive_type'],
                   comm_bits=spec_dict['comm_bits'],
                   vq_codebook_size=spec_dict.get('vq_codebook_size', 256),
                   vq_num_subspaces=spec_dict.get('vq_num_subspaces', 8),
                   vq_commitment_weight=spec_dict.get('vq_commitment_weight', 0.25))
    return build_passive(s, seed=seed), s

print('Model classes defined.')

In [ ]:
@torch.no_grad()
def extract_embeddings(ckpt_path):
    """Load checkpoint, run val set through passive, return (embeddings, labels)."""
    ckpt = _torch_load(ckpt_path)
    spec_dict = ckpt['spec']
    seed = ckpt['seed']
    passive, _ = _build_passive_for_load(spec_dict, seed=seed)
    passive.load_state_dict(ckpt['passive_state'], strict=True)
    passive = passive.to(DEVICE).eval()
    use_quantizer = (spec_dict['passive_type'] == 'vq')

    class ValDataset(Dataset):
        def __init__(self): pass
        def __len__(self): return len(val_ind)
        def __getitem__(self, i):
            idx = val_ind[i]
            return val_transforms(Image.fromarray(all_images[idx])), int(label_ints[idx])

    loader = DataLoader(ValDataset(), batch_size=128, shuffle=False,
                        num_workers=CFG.num_workers, pin_memory=True)
    embs, labs = [], []
    for imgs, lbls in loader:
        e, _, _, _ = passive(imgs.to(DEVICE), use_quantizer=use_quantizer)
        embs.append(e.cpu().numpy())
        labs.append(lbls.numpy())

    # For VQ: also report codebook utilization
    util = None
    if use_quantizer and hasattr(passive, 'quantizer'):
        counts = passive.quantizer.usage_counts.float().cpu().numpy()
        used = (counts > 0).sum(axis=1)  # per subspace
        K = passive.quantizer.K
        util = float(used.mean()) / K

    return np.vstack(embs), np.concatenate(labs), util

print('Embedding extractor defined.')

In [ ]:
# Methods to analyse — use seed 42 for everything (seed 43 would give same story)
ANALYSIS_METHODS = [
    MethodSpec(name='A_plain_vfl',      passive_type='plain',     comm_bits=40960),
    MethodSpec(name='A_proj_vfl',       passive_type='proj',      comm_bits=4096),
    MethodSpec(name='S_sign_quant',     passive_type='sign',      comm_bits=128),
    MethodSpec(name='H_vq_K64',         passive_type='vq',        comm_bits=48,
               vq_codebook_size=64,  vq_num_subspaces=8),
    MethodSpec(name='H_vq_K256',        passive_type='vq',        comm_bits=64,
               vq_codebook_size=256, vq_num_subspaces=8),
    MethodSpec(name='H_vq_M4',          passive_type='vq',        comm_bits=32,
               vq_codebook_size=256, vq_num_subspaces=4),
    MethodSpec(name='H_vq_M16',         passive_type='vq',        comm_bits=128,
               vq_codebook_size=256, vq_num_subspaces=16),
]

SEED = 42
results = {}

for spec in ANALYSIS_METHODS:
    ckpt_path = f'{CKPT_DIR}/{spec.name}_seed{SEED}.pt'
    if not os.path.exists(ckpt_path):
        print(f'[miss] {spec.name}'); continue
    print(f'Extracting: {spec.name} ...', end=' ', flush=True)
    embs, labs, util = extract_embeddings(ckpt_path)

    # Silhouette score on a subsample (full val set can be slow)
    n = min(3000, len(embs))
    rng = np.random.RandomState(0)
    idx = rng.choice(len(embs), n, replace=False)
    sil = silhouette_score(embs[idx], labs[idx], metric='cosine', random_state=0)

    results[spec.name] = {'embs': embs, 'labs': labs, 'silhouette': sil,
                          'codebook_util': util, 'comm_bits': spec.comm_bits,
                          'family': spec.passive_type}
    util_str = f'  codebook_util={util:.3f}' if util is not None else ''
    print(f'silhouette={sil:.4f}{util_str}')

print('\nDone.')

In [ ]:
# ── Silhouette bar chart ──
fig, ax = plt.subplots(figsize=(9, 4))
FAMILY_COLORS = {'plain':'#444444','proj':'#888888','sign':'#1f77b4',
                  'rand_sign':'#aec7e8','vq':'#d62728'}
names = list(results.keys())
sils  = [results[n]['silhouette'] for n in names]
cols  = [FAMILY_COLORS.get(results[n]['family'], '#000') for n in names]
bars  = ax.bar(names, sils, color=cols, edgecolor='white', linewidth=0.5)
ax.set_ylabel('Silhouette score (cosine, higher = better class separation)', fontsize=10)
ax.set_title('Val-set embedding silhouette by method', fontsize=11)
ax.set_xticklabels(names, rotation=30, ha='right', fontsize=9)
ax.axhline(0, color='grey', linewidth=0.5)
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, sils):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8)
fig.tight_layout()
fig.savefig(f'{OUT_DIR}/silhouette_by_method.pdf', dpi=150)
fig.savefig(f'{OUT_DIR}/silhouette_by_method.png', dpi=150)
plt.close(fig)
print('Saved silhouette bar chart.')

# Print table
print('\nSilhouette summary:')
print(f'{"Method":<22} {"Bits":>6} {"Silhouette":>12} {"Codebook util":>15}')
for n in names:
    r = results[n]
    u = f"{r[\"codebook_util\"]:.3f}" if r['codebook_util'] is not None else '—'
    print(f"{n:<22} {r['comm_bits']:>6} {r['silhouette']:>12.4f} {u:>15}")

with open(f'{OUT_DIR}/silhouette_table.json', 'w') as f:
    json.dump({n: {'silhouette': results[n]['silhouette'],
                   'codebook_util': results[n]['codebook_util'],
                   'comm_bits': results[n]['comm_bits']}
               for n in names}, f, indent=2)

In [ ]:
# ── t-SNE: A_plain_vfl vs H_vq_K64 side by side ──
CLASS_PALETTE = plt.cm.get_cmap('tab10', NUM_CLASSES)

def run_tsne(embs, labs, title, out_path, n_sample=2000):
    rng = np.random.RandomState(42)
    idx = rng.choice(len(embs), min(n_sample, len(embs)), replace=False)
    e_sub, l_sub = embs[idx], labs[idx]
    print(f'  Running t-SNE on {len(e_sub)} points for {title} ...')
    proj = TSNE(n_components=2, perplexity=40, random_state=42,
                n_iter=1000, verbose=0).fit_transform(e_sub)
    fig, ax = plt.subplots(figsize=(6, 5))
    for c in range(NUM_CLASSES):
        mask = l_sub == c
        if mask.sum() == 0: continue
        ax.scatter(proj[mask, 0], proj[mask, 1], s=8, alpha=0.6,
                   color=CLASS_PALETTE(c), label=CLASS_NAMES[c])
    ax.set_title(title, fontsize=11)
    ax.legend(markerscale=2, fontsize=7, loc='best')
    ax.set_xticks([]); ax.set_yticks([])
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved: {out_path}')

for method_name, label in [('A_plain_vfl', 'Plain VFL (40960 bits)'),
                             ('H_vq_K64',   'H_vq_K64 (48 bits)')]:
    if method_name not in results: continue
    run_tsne(results[method_name]['embs'],
             results[method_name]['labs'],
             title=label,
             out_path=f"{OUT_DIR}/tsne_{method_name}.png")

print('t-SNE done.')

In [ ]:
# ── Combined t-SNE side-by-side figure for paper ──
if 'A_plain_vfl' in results and 'H_vq_K64' in results:
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    for ax, (method_name, label) in zip(axes, [
        ('A_plain_vfl', 'Plain VFL  (SSIM 0.622, WACC 0.752)'),
        ('H_vq_K64',    'H_vq_K64  (SSIM 0.503, WACC 0.786)'),
    ]):
        r   = results[method_name]
        rng = np.random.RandomState(42)
        idx = rng.choice(len(r['embs']), min(2000, len(r['embs'])), replace=False)
        e_sub, l_sub = r['embs'][idx], r['labs'][idx]
        proj = TSNE(n_components=2, perplexity=40, random_state=42, n_iter=1000).fit_transform(e_sub)
        for c in range(NUM_CLASSES):
            mask = l_sub == c
            if mask.sum() == 0: continue
            ax.scatter(proj[mask, 0], proj[mask, 1], s=8, alpha=0.6,
                       color=CLASS_PALETTE(c), label=CLASS_NAMES[c])
        ax.set_title(label, fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
    axes[1].legend(markerscale=2, fontsize=7, loc='best', title='Class')
    fig.suptitle('t-SNE of transmitted val-set representations', fontsize=11)
    fig.tight_layout()
    fig.savefig(f'{OUT_DIR}/tsne_comparison.pdf', dpi=150, bbox_inches='tight')
    fig.savefig(f'{OUT_DIR}/tsne_comparison.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved combined t-SNE: {OUT_DIR}/tsne_comparison.pdf')

print(f'\nAll outputs in: {OUT_DIR}/')